In [ ]:
# Cell 1 — Install all dependencies
# Run this ONCE only. Restart kernel after this completes.

!pip install pandas xlrd openpyxl
!pip install python-docx
!pip install langchain langchain-community
!pip install faiss-cpu
!pip install sentence-transformers
!pip install groq

In [1]:
# Cell 2 — Load all Excel mock data tables

import pandas as pd
import os

# Base path — relative from main dir
BASE = r"E:\Claude_files\HSBC_Conversion\data"

# Load all 4 tables (.xls needs xlrd engine)
credit_applications = pd.read_excel(os.path.join(BASE, "credit_applications.xls"),  engine="xlrd")
basic_details = pd.read_excel(os.path.join(BASE, "basic_details.xls"),  engine="xlrd")
downstream = pd.read_excel(os.path.join(BASE, "downstream.xls"),   engine="xlrd")
multi_ratings = pd.read_excel(os.path.join(BASE, "multi_ratings.xls"),   engine="xlrd")

# Print shape and preview of each
print("=== credit_applications ===")
print(f"Shape: {credit_applications.shape}")
print(credit_applications.head())

print("\n=== basic_details ===")
print(f"Shape: {basic_details.shape}")
print(basic_details.head())

print("\n=== downstream ===")
print(f"Shape: {downstream.shape}")
print(downstream.head())

print("\n=== multi_ratings ===")
print(f"Shape: {multi_ratings.shape}")
print(multi_ratings.head())

print("\n All tables loaded successfully!")

=== credit_applications ===
Shape: (23, 5)
  relationship_id  credit_proposal_serial_no app_status app_proposal_type  \
0         1000101                     240301          L                RN   
1         1000101                     250521          A                CN   
2         1000101                     260101          A                RN   
3         1000202                     230415          A                NL   
4         1000202                     241130          L                RN   

  app_approval_dt  
0      2024-03-01  
1      2025-05-21  
2      2026-01-01  
3      2023-04-15  
4      2024-11-30  

=== basic_details ===
Shape: (23, 6)
  relationship_id customer_id  credit_proposal_serial_no  \
0         1000101     1000101                     240301   
1         1000101     1000101                     250521   
2         1000101     1000101                     260101   
3         1000202     1000202                     230415   
4         1000202     1000202       

In [2]:
# Cell 3 FIXED v3 — Fix combination bug + verify table content

import json
from docx import Document

DOCS_PATH     = r"E:\Claude_files\HSBC_Conversion\docs"
NOTEBOOK_PATH = r"E:\Claude_files\HSBC_Conversion\notebook\ratings_data_asset_developement.ipynb"

# ── Helper 1: Read docx paragraphs + tables ──────────────────
def read_docx(path):
    doc = Document(path)
    content = []

    # Read paragraphs
    for para in doc.paragraphs:
        if para.text.strip():
            content.append(para.text.strip())

    # Read tables
    for table in doc.tables:
        for row in table.rows:
            row_text = " | ".join(
                cell.text.strip()
                for cell in row.cells
                if cell.text.strip()
            )
            if row_text:
                content.append(row_text)

    return "\n".join(content)

# ── Helper 2: Read notebook ──────────────────────────────────
def read_notebook(path):
    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    chunks = []
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] in ["code", "markdown"]:
            source = "".join(cell["source"])
            if source.strip():
                label = f"[Notebook Cell {i+1} - {cell['cell_type'].upper()}]\n"
                chunks.append(label + source)
    return "\n\n".join(chunks)

# ── Load all sources ─────────────────────────────────────────
print("Loading BRD...")
brd_text = read_docx(f"{DOCS_PATH}\\BRD_CreditRiskPipeline.docx")
print(f"BRD loaded     — {len(brd_text):,} characters, "
      f"{len(brd_text.splitlines())} lines")

print("Loading ERD...")
erd_text = read_docx(f"{DOCS_PATH}\\ERD_CreditRiskPipeline.docx")
print(f"ERD loaded     — {len(erd_text):,} characters, "
      f"{len(erd_text.splitlines())} lines")

print("Loading Notebook...")
notebook_text = read_notebook(NOTEBOOK_PATH)
print(f"Notebook loaded — {len(notebook_text):,} characters, "
      f"{len(notebook_text.splitlines())} lines")

# ── Combine — fixed version ──────────────────────────────────
separator = "\n" + "="*60 + "\n"

knowledge_base = separator.join([
    "=== BRD: Business Requirements Document ===\n" + brd_text,
    "=== ERD: Entity Relationship Document ===\n"   + erd_text,
    "=== JUPYTER NOTEBOOK: CFC_PD_ECB ===\n"        + notebook_text
])

print(f"\nTotal knowledge base: {len(knowledge_base):,} characters")

# ── Verify BRD table content is present ─────────────────────
print("\n=== Checking BRD table rows are captured ===")
brd_lines = brd_text.splitlines()
table_lines = [l for l in brd_lines if "|" in l]
print(f"Table rows found in BRD: {len(table_lines)}")
print("\nSample table rows:")
for line in table_lines[:5]:
    print(f"  {line}")

print("\n=== Checking ERD table rows are captured ===")
erd_lines = erd_text.splitlines()
erd_table_lines = [l for l in erd_lines if "|" in l]
print(f"Table rows found in ERD: {len(erd_table_lines)}")
print("\nSample table rows:")
for line in erd_table_lines[:5]:
    print(f"  {line}")

Loading BRD...
BRD loaded     — 8,488 characters, 80 lines
Loading ERD...
ERD loaded     — 5,228 characters, 82 lines
Loading Notebook...
Notebook loaded — 4,835 characters, 128 lines

Total knowledge base: 18,798 characters

=== Checking BRD table rows are captured ===
Table rows found in BRD: 55

Sample table rows:
  Version 1.0  |  March 2026
  Credit Risk Data Pipeline  |  BRD Version 1.0  |  March 2026  |  Internal Use Only
  System | Join Type | Join Keys | Purpose
  credit_applications | Base / Filtered | relationship_id + credit_proposal_serial_no | Credit proposals — source of truth for proposal lifecycle
  basic_details | INNER JOIN | relationship_id + customer_id + credit_proposal_serial_no | Baseline risk ratings (final_crr, PD, scorecard) — fallback source

=== Checking ERD table rows are captured ===
Table rows found in ERD: 63

Sample table rows:
  Version 1.0  |  March 2026
  |
  |── INNER JOIN ──> basic_details   (on relationship_id + credit_proposal_serial_no)
  |    

In [3]:
# Cell 4 — Build FAISS Vector Store (replaces ChromaDB)

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Step 4a: Split into chunks ───────────────────────────────
print("Splitting documents into chunks...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.create_documents([knowledge_base])
print(f"Total chunks created: {len(chunks)}")

# ── Step 4b: Load embedding model ───────────────────────────
print("\nLoading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
print("Embedding model loaded!")

# ── Step 4c: Build FAISS vector store ───────────────────────
print("\nBuilding FAISS vector store...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"FAISS vector store built with {len(chunks)} chunks!")

# ── Step 4d: Save locally so you don't rebuild every time ───
FAISS_PATH = r"E:\Claude_files\HSBC_Conversion\faiss_index"
vectorstore.save_local(FAISS_PATH)
print(f"Vector store saved to: {FAISS_PATH}")

# ── Step 4e: Sanity test ─────────────────────────────────────
print("\n--- Sanity Test ---")
test_query = "Why is final_crr null?"
results = vectorstore.similarity_search(test_query, k=2)

print(f"Test query: '{test_query}'")
print(f"Top result preview:\n{results[0].page_content[:300]}")
print("\nRAG is working! Ready for Step 5.")

Splitting documents into chunks...
Total chunks created: 47

Loading embedding model...


C:\Users\amit pc\AppData\Local\Temp\ipykernel_23588\3285308850.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(



Embedding model loaded!

Building FAISS vector store...
FAISS vector store built with 47 chunks!
Vector store saved to: E:\Claude_files\HSBC_Conversion\faiss_index

--- Sanity Test ---
Test query: 'Why is final_crr null?'
Top result preview:
final_crr | COALESCE(final_crr_multi_ratings, final_crr_basic_details) | crr_source | YES — if both NULL
final_pd | COALESCE(final_pd_multi_ratings, final_pd_basic_details) | crr_source | YES — if both NULL
internal_score_card | COALESCE(internal_score_card_multi_ratings, internal_score_card_basic_d

RAG is working! Ready for Step 5.


In [5]:
# Cell 5 — Connect to Groq LLM
from groq import Groq

# ── Your Groq API Key ────────────────────────────────────────
GROQ_API_KEY = ''  # ← paste your key here

# ── Initialize Groq client ───────────────────────────────────
client = Groq(api_key=GROQ_API_KEY)

# ── LLM call function ────────────────────────────────────────
def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",   # best free model on Groq
            messages=[
                {
                    "role": "system",
                    "content": """You are an intelligent debugging assistant 
                    for the PD Credit Risk Model. You help Finance 
                    teams understand why specific attribute values are null, 
                    incorrect, or unexpected. Always be precise, reference 
                    table names and column names in your answers."""
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1,   # low = more factual, less creative
            max_tokens=1000
        )
        return response.choices[0].message.content
    
    except Exception as e:
        return f" LLM call failed: {str(e)}"

# ── Sanity test ──────────────────────────────────────────────
print("Testing Groq connection...")

test_response = call_llm(
    "In the ratings data asset developement, what happens when "
    "both multiratings and basic_details values are null for final_crr?"
)

print(" Groq connected successfully!")
print("\n--- Test Response ---")
print(test_response)

Testing Groq connection...
 Groq connected successfully!

--- Test Response ---
When both `multiratings` and `basic_details` values are null for `final_crr` in the ratings data asset development, it indicates that there is a lack of necessary information to determine the credit risk rating.

In the `credit_risk_ratings` table, the `final_crr` column is typically calculated based on the values from the `multiratings` and `basic_details` tables. If both of these values are null, it suggests that there is a data gap or missing information that is required to accurately calculate the `final_crr`.

To troubleshoot this issue, I would recommend checking the following:

1. **Data ingestion pipeline**: Verify that the data ingestion pipeline is correctly loading data from the source systems into the `multiratings` and `basic_details` tables.
2. **Data quality checks**: Perform data quality checks on the `multiratings` and `basic_details` tables to identify any data inconsistencies or missing v

In [14]:
# Cell 6 — PDHBC Debug Agent
# Chains RAG + Pandas data lookup + Groq LLM together

import re
import pandas as pd

# ── Helper: Get RAG context from vector store ────────────────
def get_rag_context(query, k=4):
    results = vectorstore.similarity_search(query, k=k)
    return "\n\n".join([r.page_content for r in results])

# ── Helper: Look up customer across all tables ───────────────
def lookup_customer(relationship_id=None, serial_no=None):
    result = {}

    # credit_applications lookup
    credit_applications_match = credit_applications.copy()
    if relationship_id:
        credit_applications_match = credit_applications_match[
            credit_applications_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    if serial_no:
        credit_applications_match = credit_applications_match[
            credit_applications_match["credit_proposal_serial_no"].astype(str) == str(serial_no)
        ]
    result["credit_applications"] = credit_applications_match.to_string() if not credit_applications_match.empty else "No record found"

    # basic_details lookup
    basic_details_match = basic_details.copy()
    if relationship_id:
        basic_details_match = basic_details_match[
            basic_details_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["basic_details"] = basic_details_match.to_string() if not basic_details_match.empty else "No record found"

    # multi_ratings lookup
    multi_ratings_match = multi_ratings.copy()
    if relationship_id:
        multi_ratings_match = multi_ratings_match[
            multi_ratings_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["multi_ratings"] = multi_ratings_match.to_string() if not multi_ratings_match.empty else "No record found"

    # downstream lookup
    downstream_match = downstream.copy()
    if relationship_id:
        downstream_match = downstream_match[
            downstream_match["relationship_id"].astype(str) == str(relationship_id)
        ]
    result["downstream"] = downstream_match.to_string() if not downstream_match.empty else "No record found"

    return result

# ── Helper: Check nulls for a relationship ───────────────────
def check_nulls(relationship_id):
    issues = []

    # Check multi_ratings
    multi_ratings_row = multi_ratings[multi_ratings["relationship_id"].astype(str) == str(relationship_id)]
    if multi_ratings_row.empty:
        issues.append(" multi_ratings: No record at all — BASDP fallback will be used")
    else:
        for col in ["final_crr_mrdp", "final_pd_mrdp", "internal_score_card_mrdp"]:
            if col in multi_ratings_row.columns and multi_ratings_row[col].isnull().any():
                issues.append(f"  MRDP.{col} is NULL")

    # Check basic_details
    basic_details_row = basic_details[basic_details["relationship_id"].astype(str) == str(relationship_id)]
    if basic_details_row.empty:
        issues.append("basic_details: No record — final_crr will be MISSING")
    else:
        for col in ["final_crr_basd", "final_pd_basd", "internal_score_card_basd"]:
            if col in basic_details_row.columns and basic_details_row[col].isnull().any():
                issues.append(f"basic_details.{col} is NULL")

    # Check downstream
    downstream_row = downstream[downstream["relationship_id"].astype(str) == str(relationship_id)]
    if downstream_row.empty:
        issues.append("downstream: No record — local_customer_id will be NULL")

    # Check credit_applications filters
    credit_applications_row = credit_applications[credit_applications["relationship_id"].astype(str) == str(relationship_id)]
    if not credit_applications_row.empty:
        cn_rows = credit_applications_row[credit_applications_row["app_proposal_type"] == "CN"]
        lapsed_rows = credit_applications_row[credit_applications_row["app_status"] == "L"]
        if not cn_rows.empty:
            issues.append("credit_applications: CN proposal type exists — will be excluded by filter")
        if not lapsed_rows.empty:
            issues.append("credit_applications: Lapsed (L) proposals exist — will be excluded by filter")

    return issues if issues else ["No null issues detected"]

# ── Extract IDs from natural language query ──────────────────
def extract_ids(query):
    rel_id  = None
    serial  = None
    lcl_id  = None

    # Match local_customer_id: LCL-IN-xxxx, LCL-FR-xxxx, LCL-DE-xxxx
    lcl_match = re.search(
        r'\bLCL-(IN|FR|DE)-\d+\b', 
        query, re.IGNORECASE
    )
    if lcl_match:
        lcl_id = lcl_match.group().upper()

    # Match relationship_id: 1000101, FR100201, 20100374
    rel_match = re.search(
        r'\b(FR\d+|20\d{5,}|10\d{5,})\b',
        query, re.IGNORECASE
    )
    if rel_match:
        rel_id = rel_match.group()

    # Match serial number: 6 digit numbers like 240301
    serial_match = re.search(r'\b(\d{6})\b', query)
    if serial_match and rel_id:
        if serial_match.group() != rel_id:
            serial = serial_match.group()

    return rel_id, serial, lcl_id  # ← now returns 3 values

# ── Helper: Reverse lookup from local_customer_id ────────────
def resolve_local_customer_id(lcl_id):
    """
    Given a local_customer_id like LCL-IN-5530,
    find the relationship_id and customer_id from WDRP
    then trace through all tables
    """
    print(f"\n Resolving local_customer_id: {lcl_id}")
    
    # Step 1: Find in WDRP
    downstream_match = downstream[
        downstream["local_customer_id"].astype(str) == str(lcl_id)
    ]
    
    if downstream_match.empty:
        return None, None, f"'{lcl_id}' not found in downstream table"
    
    relationship_id = str(downstream_match["relationship_id"].values[0])
    customer_id     = str(downstream_match["customer_id"].values[0])
    
    print(f"Found in downstream:")
    print(f"local_customer_id → {lcl_id}")
    print(f"relationship_id   → {relationship_id}")
    print(f"customer_id       → {customer_id}")
    
    return relationship_id, customer_id, None

# ── Helper: Apply credit_applications base filters + find latest proposal ──
def get_latest_valid_proposal(relationship_id):
    """
    Apply all credit_applications base filters:
    1. app_status = 'A'     (approved only)
    2. app_proposal_type != 'CN'  (exclude change-in-terms)
    3. MAX(app_approval_dt) (latest wins)
    """
    credit_applications_match = credit_applications[
        credit_applications["relationship_id"].astype(str) == str(relationship_id)
    ].copy()
    
    if credit_applications_match.empty:
        return None, "No records found in credit_applications for this relationship_id"
    
    print(f"\n credit_applications records before filters: {len(credit_applications_match)}")
    print(credit_applications_match[["relationship_id", 
                        "credit_proposal_serial_no",
                        "app_status", 
                        "app_proposal_type", 
                        "app_approval_dt"]].to_string())
    
    # Filter 1: Only approved
    approved = credit_applications_match[credit_applications_match["app_status"] == "A"]
    print(f"\nAfter app_status='A' filter: {len(approved)} records")
    
    # Filter 2: Exclude CN
    non_cn = approved[approved["app_proposal_type"] != "CN"]
    print(f"After excluding CN proposals: {len(non_cn)} records")
    
    if non_cn.empty:
        return None, "No valid proposals after filters — all are CN or non-Approved"
    
    # Filter 3: Latest approval date
    non_cn = non_cn.copy()
    non_cn["app_approval_dt"] = pd.to_datetime(
        non_cn["app_approval_dt"], dayfirst=True, errors="coerce"
    )
    latest = non_cn.sort_values("app_approval_dt", ascending=False).iloc[0]
    
    print(f"\nLatest valid proposal selected:")
    print(f"credit_proposal_serial_no : {latest['credit_proposal_serial_no']}")
    print(f"app_proposal_type         : {latest['app_proposal_type']}")
    print(f"app_approval_dt           : {latest['app_approval_dt']}")
    
    return latest, None

# ── MAIN AGENT FUNCTION ──────────────────────────────────────
def debug_agent(finance_query):
    print(f"\n{'='*60}")
    print(f"QUERY: {finance_query}")
    print('='*60)

    # Step 1: RAG context
    print("\n Step 1: Retrieving relevant logic from BRD/ERD...")
    rag_context = get_rag_context(finance_query)
    print("RAG context retrieved")

    # Step 2: Extract IDs
    print("\n Step 2: Extracting IDs from query...")
    rel_id, serial, lcl_id = extract_ids(finance_query)

    # Step 2b: If LCL ID found, resolve to relationship_id first
    resolution_context = ""
    latest_proposal    = None

    if lcl_id:
        print(f"Local Customer ID found: {lcl_id}")
        rel_id, cust_id, error = resolve_local_customer_id(lcl_id)
        
        if error:
            resolution_context = error
        else:
            resolution_context = f"""
                                      Local Customer ID Resolution:
                                      LCL ID          : {lcl_id}
                                      relationship_id : {rel_id}
                                      customer_id     : {cust_id}
                                      Source Table    : downstream
                                    """
            # Now find latest valid proposal
            print("\n Applying credit_applications base filters...")
            latest_proposal, prop_error = get_latest_valid_proposal(rel_id)
            
            if prop_error:
                resolution_context += f"\nProposal lookup: {prop_error}"
            else:
                resolution_context += f"""
                                          Latest Valid Proposal (after credit_applications filters):
                                          credit_proposal_serial_no : {latest_proposal['credit_proposal_serial_no']}
                                          app_proposal_type         : {latest_proposal['app_proposal_type']}
                                          app_approval_dt           : {latest_proposal['app_approval_dt']}
                                          Filters applied           : app_status='A', app_proposal_type!='CN', MAX(app_approval_dt)
                                        """

    elif rel_id:
        print(f"Relationship ID found: {rel_id}")
        if serial:
            print(f" Serial Number found: {serial}")
    else:
        print(" No ID found — answering from documents only")

    # Step 3: Data lookup
    data_context = ""
    null_issues  = []

    if rel_id:
        print("\n Step 3: Looking up data across all tables...")
        table_data  = lookup_customer(relationship_id=rel_id, serial_no=serial)
        null_issues = check_nulls(rel_id)

        data_context = f"""
                            {resolution_context}
                            
                            --- credit_applications Data ---
                            {table_data['credit_applications']}
                            
                            --- basic_details Data ---
                            {table_data['basic_details']}
                            
                            --- multi_ratings Data ---
                            {table_data['multi_ratings']}
                            
                            --- downstream Data ---
                            {table_data['downstream']}
                            
                            --- Null/Issue Check ---
                            {chr(10).join(null_issues)}
                        """
        print("Data lookup complete")
        print("\n Issues detected:")
        for issue in null_issues:
            print(f"   {issue}")
    else:
        data_context = resolution_context or \
                       "No specific ID provided — answering from documents only."

    # Step 4: Build prompt
    print("\n Step 4: Building prompt for LLM...")
    prompt = f"""
                You are an expert debugging assistant for the PDHBC Credit Risk Pipeline at HSBC.
                Use ONLY the information provided below to answer. Do not invent table names or columns.
                
                Finance Team Query:
                {finance_query}
                
                Relevant Logic from BRD/ERD Documents:
                {rag_context}
                
                Live Data from Tables:
                {data_context}
                
                Please provide a structured answer with:
                1. Direct answer to the query
                2. Root cause (which table/column is causing the issue)
                3. Which pipeline rule applies (e.g. COALESCE, filter, fallback)
                4. Recommended next step for the Finance team
                """

    # Step 5: Call LLM
    print("\n Step 5: Calling Groq LLM...")
    answer = call_llm(prompt)

    print("\n" + "="*60)
    print("FINAL ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)

    return None

In [17]:
#Test the agent with a real Finance query

finance_query = "Why is the local_customer_id, customer-id for relationship 1000101 that pass our credit_applications filters?"
debug_agent(finance_query)


QUERY: Why is the local_customer_id, customer-id for relationship 1000101 that pass our credit_applications filters?

 Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

 Step 2: Extracting IDs from query...
Relationship ID found: 1000101

 Step 3: Looking up data across all tables...
Data lookup complete

 Issues detected:
   credit_applications: CN proposal type exists — will be excluded by filter
   credit_applications: Lapsed (L) proposals exist — will be excluded by filter

 Step 4: Building prompt for LLM...

 Step 5: Calling Groq LLM...

FINAL ANSWER:
### 1. Direct Answer to the Query
The `local_customer_id` for relationship `1000101` that passes the credit applications filters is `LCL-IN-4401`. This is because, after applying the filters to exclude `CN` proposal types and `L` (lapsed) proposals, the remaining approved (`A`) proposals for relationship `1000101` are associated with this `local_customer_id` in the `downstream` table.

### 2. Root Cause
The r

In [18]:
finance_query = "For LCL-IN-5530 what is the latest credit proposal that satisfies the base filters?"
debug_agent(finance_query)


QUERY: For LCL-IN-5530 what is the latest credit proposal that satisfies the base filters?

 Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

 Step 2: Extracting IDs from query...
Local Customer ID found: LCL-IN-5530

 Resolving local_customer_id: LCL-IN-5530
Found in downstream:
local_customer_id → LCL-IN-5530
relationship_id   → 1000202
customer_id       → 1000202

 Applying credit_applications base filters...

 credit_applications records before filters: 2
  relationship_id  credit_proposal_serial_no app_status app_proposal_type app_approval_dt
3         1000202                     230415          A                NL      2023-04-15
4         1000202                     241130          L                RN      2024-11-30

After app_status='A' filter: 1 records
After excluding CN proposals: 1 records

Latest valid proposal selected:
credit_proposal_serial_no : 230415
app_proposal_type         : NL
app_approval_dt           : 2023-04-15 00:00:00

 Step 3: Look

In [19]:

finance_query = "Why local_customer_id is blank for relation 1000305"
debug_agent(finance_query)


QUERY: Why local_customer_id is blank for relation 1000305

 Step 1: Retrieving relevant logic from BRD/ERD...
RAG context retrieved

 Step 2: Extracting IDs from query...
Relationship ID found: 1000305

 Step 3: Looking up data across all tables...
Data lookup complete

 Issues detected:
   downstream: No record — local_customer_id will be NULL
   credit_applications: CN proposal type exists — will be excluded by filter

 Step 4: Building prompt for LLM...

 Step 5: Calling Groq LLM...

FINAL ANSWER:
### 1. Direct Answer to the Query
The `local_customer_id` is blank for relation `1000305` because there is no record found in the `downstream` table that matches the `relationship_id` and `customer_id` of `1000305`.

### 2. Root Cause
The root cause of the issue is that the `downstream` table does not contain a record for `relationship_id` `1000305`, which is the table that provides the `local_customer_id` value. Specifically, the `LEFT JOIN` operation on `relationship_id` and `customer_